<a href="https://colab.research.google.com/github/urshishir6/Credit-Card-Fraud-Detection-Project-1/blob/main/01_EDA_and_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# [Team Lead: Chaulagai Shishir] - Environment Setup & Reproducibility
!pip install -q imbalanced-learn xgboost seaborn matplotlib scikit-learn pandas numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Set global seed for absolute reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment setup complete. All libraries imported.")

Environment setup complete. All libraries imported.


## 1. Data Loading and Exploratory Data Analysis (EDA)
**Lead: Deula Ritika (Acquisition) & Malla Abiraj (EDA)**  
* **Initial Inspection:** Verifying data types and ensuring zero missing values.
* **Target Distribution Plot:** Demonstrating the extreme 99.8% vs 0.17% class imbalance.
* **Feature Distribution:** Plotting histograms of `Time` and `Amount` to visualize extreme skewness and outliers before applying any scaling.

In [7]:
# 1.1 Download and Load Dataset from Web
import os
import urllib.request

# Direct URL to the raw, unzipped creditcard.csv
url = "https://raw.githubusercontent.com/kartik-parmar/Credit-Card-Dataset/main/creditcard.csv"
filename = "creditcard.csv"

if not os.path.exists(filename):
    print(f"Downloading dataset from: {url}...")
    try:
        req = urllib.request.Request(
            url,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
        with urllib.request.urlopen(req) as response, open(filename, 'wb') as out_file:
            out_file.write(response.read())
        print("Download and write complete!")
    except Exception as e:
        raise RuntimeError(f"Failed to download dataset: {e}. Please check your internet connection.")

try:
    df = pd.read_csv('creditcard.csv')
    print("Dataset 'creditcard.csv' loaded successfully!\n")
except Exception as e:
    raise RuntimeError(f"ERROR loading dataset: {e}")

# 1.2 Initial Inspection (info and missing values)
print("--- Data Information ---")
df.info()
print("\n--- Missing Values Check ---")
missing_vals = df.isnull().sum().sum()
print(f"Total Missing Values across all columns: {missing_vals}\n")

# 1.3 Exploratory Data Analysis (Visualizations)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Target Distribution Plot (Bar Plot)
class_counts = df['Class'].value_counts(normalize=True) * 100
sns.countplot(x='Class', data=df, ax=axes[0], palette='Set2')
axes[0].set_title(f'Target Class Imbalance\n(Legitimate: {class_counts[0]:.3f}%, Fraud: {class_counts[1]:.3f}%)')
axes[0].set_yscale('log') # Log scale is essential here to even see the fraud class
axes[0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])

# Feature Distribution Plot (Time)
sns.histplot(df['Time'], bins=50, ax=axes[1], color='steelblue', kde=False)
axes[1].set_title('Distribution of Transaction Time\n(Bimodal, Unscaled)')
axes[1].set_xlabel('Time (seconds)')

# Feature Distribution Plot (Amount)
sns.histplot(df['Amount'], bins=50, ax=axes[2], color='darkred', kde=False)
axes[2].set_title('Distribution of Transaction Amount\n(Extreme Skewness & Outliers)')
axes[2].set_xlabel('Amount ($)')
axes[2].set_yscale('log') # Log scale to visualize extreme outliers

plt.tight_layout()
plt.show()

RuntimeError: Failed to download dataset: HTTP Error 404: Not Found. Please check your internet connection.

## 2. Train/Test Separation (Leakage Prevention)
**Lead: KC hikalpit**  
We isolate the target and perform a stratified split. This ensures that the rare fraud cases are proportionally distributed. **Crucially, this is done before any scaling or SMOTE to prevent data leakage.**

In [ ]:
# Isolate Features (X) and Target (y)
X = df.drop('Class', axis=1)
y = df['Class']

# Stratified Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("--- Stratified Split Complete ---")
print(f"Training Data: {X_train.shape[0]} rows (Frauds: {sum(y_train==1)})")
print(f"Testing Data:  {X_test.shape[0]} rows (Frauds: {sum(y_test==1)})")

## 3. Feature Scaling
**Lead: Budhathoki Sandesh**  
Applying `RobustScaler` to handle the extreme outliers in the financial data. We fit it *only* on the `X_train` data for `Time` and `Amount`, leaving `V1-V28` untouched.

In [ ]:
# Initialize Scaler
scaler = RobustScaler()

# Fit on training set ONLY, Transform both
X_train[['Time', 'Amount']] = scaler.fit_transform(X_train[['Time', 'Amount']])
X_test[['Time', 'Amount']] = scaler.transform(X_test[['Time', 'Amount']])

print("RobustScaler successfully applied to 'Time' and 'Amount'. V1-V28 remain untouched.")

## 4 & 5. Imbalance Handling, Pipeline & Model Training
**Lead: Rawat Nabraj & Thada Magar Mansi**  
1. **Naive Benchmark:** An array of zeros to prove accuracy is a misleading metric.
2. **Interpretable Model (Logistic Regression):** Uses `imblearn.pipeline.Pipeline` with SMOTE.
3. **Advanced Models (Random Forest & XGBoost):** Using algorithm-level class weights (`class_weight='balanced'` and `scale_pos_weight`) to handle imbalance without synthetic generation.

In [ ]:
# 1. Naive Benchmark (Literal array of zeros)
print("1. Generating Naive Benchmark (Predicting all zeros)...")
y_pred_naive = np.zeros(len(y_test))
y_probs_naive = np.zeros(len(y_test))

# 2. Interpretable Model (Logistic Regression + SMOTE Pipeline)
print("2. Training Logistic Regression via Imblearn SMOTE Pipeline...")
pipeline_lr = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('logreg', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# Extra feature for perfection: Demonstrate cross-validation correctly using the pipeline
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(pipeline_lr, X_train, y_train, cv=cv, scoring='average_precision')
print(f"   -> Pipeline CV Mean PR-AUC: {cv_scores.mean():.4f} (Validates no leakage during CV)")

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
y_probs_lr = pipeline_lr.predict_proba(X_test)[:, 1]

# 3. Candidate Model 2 (Random Forest)
print("3. Training Random Forest (class_weight='balanced')...")
rf = RandomForestClassifier(n_estimators=50, max_depth=10, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_probs_rf = rf.predict_proba(X_test)[:, 1]

# 4. Candidate Model 3 (XGBoost)
print("4. Training XGBoost (scale_pos_weight)...")
scale_pos = sum(y_train==0) / sum(y_train==1)
xgb = XGBClassifier(scale_pos_weight=scale_pos, random_state=RANDOM_STATE, eval_metric='aucpr')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
y_probs_xgb = xgb.predict_proba(X_test)[:, 1]

print("\nAll models trained successfully!")

## 6. Evaluation and Metrics Generation
**Lead: KC hikalpit**  
Because standard accuracy is invalid for 99.8% imbalanced data, we systematically compare models using Precision, Recall, F1-Score, and primarily **PR-AUC**.

In [ ]:
# Function to calculate metrics safely
def get_metrics(y_true, y_pred, y_probs):
    return {
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'PR-AUC': average_precision_score(y_true, y_probs)
    }

# Compile results into a Dictionary
results = {
    'Naive Benchmark (Zeros)': get_metrics(y_test, y_pred_naive, y_probs_naive),
    'Logistic Regression (SMOTE)': get_metrics(y_test, y_pred_lr, y_probs_lr),
    'Random Forest (Balanced)': get_metrics(y_test, y_pred_rf, y_probs_rf),
    'XGBoost': get_metrics(y_test, y_pred_xgb, y_probs_xgb)
}

# Convert to Pandas DataFrame for a clean metric table
results_df = pd.DataFrame(results).T
print("=== FINAL MODEL COMPARISON METRIC TABLE ===")
display(results_df.sort_values(by='PR-AUC', ascending=False).style.highlight_max(color='lightgreen', axis=0))

## 7. Model Interpretation and Error Analysis
**Lead: Aayusha Bista**  
Visually analyzing errors via Confusion Matrices, extracting Logistic Regression coefficients, and plotting Tree-based Feature Importances.

In [ ]:
# 7.1 Error Analysis (Confusion Matrix for XGBoost vs Logistic Regression)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Legitimate', 'Fraud'])
disp_lr.plot(cmap='Blues', ax=axes[0], values_format='d', colorbar=False)
axes[0].set_title('Logistic Regression (SMOTE)\nHigh Recall, High False Positives')

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
disp_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=['Legitimate', 'Fraud'])
disp_xgb.plot(cmap='Reds', ax=axes[1], values_format='d', colorbar=False)
axes[1].set_title('XGBoost\nBalanced Error Rate')

plt.tight_layout()
plt.show()

# 7.2 Interpret Logistic Regression (Coefficients via imblearn pipeline)
logreg_model = pipeline_lr.named_steps['logreg']
coefficients = logreg_model.coef_[0]
coef_indices = np.argsort(np.abs(coefficients))[::-1][:10]

plt.figure(figsize=(10, 5))
plt.bar(range(10), coefficients[coef_indices], color=['red' if c < 0 else 'green' for c in coefficients[coef_indices]])
plt.xticks(range(10), X.columns[coef_indices], rotation=45)
plt.axhline(0, color='black', linewidth=1)
plt.title("Logistic Regression: Top 10 Drivers of Fraud (Coefficients)")
plt.ylabel("Coefficient Value")
plt.show()

# 7.3 Interpret Tree Models (XGBoost Feature Importances)
importances = xgb.feature_importances_
indices = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(10, 5))
plt.bar(range(10), importances[indices], color='darkorange')
plt.xticks(range(10), X.columns[indices], rotation=45)
plt.title("XGBoost: Top 10 Feature Importances")
plt.ylabel("Relative Importance")
plt.show()

## 8. Final Recommendation, Limitations & Responsible Use
**Team Conclusion:**

1. **Selected Model:** **XGBoost** is our recommended model. As shown in the metric table, it overwhelmingly beat the baseline and Logistic Regression with the highest PR-AUC and F1-Score.
2. **Error Analysis Trade-Off:** The Confusion Matrix reveals that Logistic Regression + SMOTE caught most frauds but generated thousands of False Positives. XGBoost drastically reduced these False Positives while still catching the majority of frauds.
3. **Data Limitations:** Because features `V1` to `V28` are PCA-transformed, we cannot directly explain *what* these top features (like `V14` or `V4`) represent in the real world (e.g., location, merchant type).
4. **Responsible AI Use:** False positives block legitimate customer credit cards, causing severe customer dissatisfaction. Therefore, this model must be used strictly as a **decision-support tool**—flagging high-risk transactions for a human fraud analyst's review, rather than automating immediate account suspensions.